# this is the notebook version of 20_hyperalignment_run.py to test and see if it is working

In [1]:
# using fmralign package to do template-based (not pairwise) hyperalignment
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
from collections import defaultdict
from itertools import combinations
from scipy.linalg import orthogonal_procrustes
from scipy.stats import pearsonr
from scipy.stats import ttest_1samp, linregress
from statsmodels.stats.multitest import multipletests
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.preprocessing import LabelEncoder
from nilearn.maskers import NiftiMasker
from nilearn.image import concat_imgs
from nilearn import plotting
from templateflow import api
from fmralign import GroupAlignment, PairwiseAlignment
from fmralign.embeddings.parcellation import get_labels
from fmralign.metrics import score_voxelwise
sys.path.insert(0, str(Path.cwd()))
from utils import (
    TASKS, CONTRASTS, SUBJECTS, SESSIONS, ENCOUNTERS,
    build_first_level_contrast_map_path, is_valid_contrast_map, clean_z_map_data,
    convert_to_regular_dict, create_smor_atlas,load_smor_atlas, load_schaefer_atlas, cleanup_memory
)
from config import OUTPUT_DIRS, BASE_DIR

/oak/stanford/groups/russpold/users/nklevak/network_second_modeling/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 3.2.1'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
ANALYSIS_DIR   = Path.cwd()
DATA_DIR       = ANALYSIS_DIR / OUTPUT_DIRS["smor"]
OUTPUT_DIR     = ANALYSIS_DIR / "hyperalignment_fmralign_results" / "discovery_sample"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

N_PARCELS        = 429
ENCOUNTER_CENTER = 3.0   # encounters 1-5 → centered: -2,-1,0,1,2
RT_CONTRAST = "response_time" # going to exclude this one for now

In [3]:
#########################################################################################
# using the betas not the z-scores
def load_parcel_dict(output_ending = "_betas", date_updated = "0111") -> dict:
    """
    Merge all 3 parcellated beta pkl files into one dict.
    Structure: parcel_dict[subject][task][contrast][encounter]
               → DataFrame with columns ['region', 'activation']
    """
    # other output_ending option is: "_z_scored"

    parcel_dict = {}
    mean_filename = f"{main_files['mean']}_{date_updated}" # fname = DATA_DIR / f"discovery_parcel_indiv_mean_updated_{date_updated}_{idx}_betas.pkl"

    for idx in [1, 2, 3]:
        fname = DATA_DIR / f"{mean_filename}_{idx}_betas.pkl"
        if not fname.exists():
            warnings.warn(f"Missing pkl file: {fname}")
            continue
        with open(fname, "rb") as f:
            chunk = pickle.load(f)
        parcel_dict.update(chunk)
        print(f"  Loaded {fname.name} → {len(chunk)} subjects")
    print(f"Total subjects loaded: {list(parcel_dict.keys())}")
    return parcel_dict

def get_activation_vector(parcel_dict, subject, task, contrast, encounter):
    """
    Extract the 429-dim activation vector for one subject/task/contrast/encounter.
    Returns None if not present or wrong length.
    """
    try:
        df = parcel_dict[subject][task][contrast][encounter]
        if df is None:
            return None
        v = df.set_index("region")["activation"].values.astype(float)
        return v if len(v) == N_PARCELS else None
    except (KeyError, AttributeError):
        return None

def build_contrast_matrix(parcel_dict: dict) -> tuple[dict, list]:
    """
    For all subjects build a ((N_sessions x N_shared_contrasts) × 429) matrix.

    Only contrasts/sessions present for ALL subjects are included (shared rows guarantee
    that Procrustes operates on semantically identical rows across subjects).
    response_time contrasts are excluded from the alignment signal.

    Remove any sessions that are not done by all subjects.

    Returns:
        matrices : {subject -> np.ndarray (N_shared × 429)}
        row_labels: list of "task__contrast" strings (same order for all subjects)
    """

    print("\n=== STEP 1: Building contrast matrices ===")

    # Collect all available (task, contrast, enc) tuples per subject
    available: dict[str, set] = {s: set() for s in SUBJECTS}
    for subj in SUBJECTS:
        for task in TASKS:
            for contrast in CONTRASTS[task]:
                if contrast == RT_CONTRAST:
                    continue
                for enc in ENCOUNTERS:
                    v = get_activation_vector(parcel_dict, subj, task, contrast, enc)
                    if v is not None:
                        available[subj].add((task, contrast, enc))
                        # no break — collect ALL valid encounters

    # Shared rows = intersection across all subjects
    shared = set.intersection(*available.values())
    # Sort deterministically: by task order, then contrast order, then encounter
    task_order = {t: i for i, t in enumerate(TASKS)}
    row_labels_sorted = sorted(
        shared,
        key=lambda tce: (task_order[tce[0]], CONTRASTS[tce[0]].index(tce[1]), tce[2])
    )
    row_labels = [f"{t}__{c}__{e}" for t, c, e in row_labels_sorted]

    print(f"Shared (task, contrast, encounter) tuples across all subjects: {len(row_labels_sorted)}")
    for rc in row_labels_sorted:
        print(f"  {rc[0]}/{rc[1]}/enc{rc[2]}")

    # Build per-subject matrices
    matrices = {}
    for subj in SUBJECTS:
        rows = []
        for task, contrast, enc in row_labels_sorted:
            v = get_activation_vector(parcel_dict, subj, task, contrast, enc)
            rows.append(v)
        mat = np.array(rows)                           # (N_shared × 429)
        matrices[subj] = mat
        print(f"  {subj}: contrast matrix shape {mat.shape}")

    return matrices, row_labels

##########################################################################
# load raw beta NIfTI maps (not parcellated)
def load_beta_maps() -> dict:
    """
    Load raw beta NIfTI maps for all subjects/tasks/contrasts/encounters.

    Mirrors the notebook approach in 15_cleaned_parcellate_individuals.ipynb:
    iterates over SESSIONS per subject/task/contrast and counts encounters
    independently for each combination.

    Returns:
        maps : {task: {contrast: {subject: {encounter_str: NIfTI image}}}}
                encounter_str is zero-padded, e.g. '01', '02', ..., '05'
    """
    maps = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

    for task in TASKS:
        for contrast in CONTRASTS[task]:
            for subject in SUBJECTS:
                overall_encounter_count = 0
                for session in SESSIONS:
                    contrast_map_path = build_first_level_contrast_map_path(
                        BASE_DIR, subject, session, task, contrast
                    )
                    if os.path.exists(contrast_map_path):
                        enc_key = f"{overall_encounter_count + 1:02d}"
                        maps[task][contrast][subject][enc_key] = nib.load(contrast_map_path)
                        overall_encounter_count += 1

    # Report what was loaded
    for task in TASKS:
        for contrast in CONTRASTS[task]:
            for subject in SUBJECTS:
                n_enc = len(maps[task][contrast].get(subject, {}))
                if n_enc > 0:
                    print(f"  {subject} | {task} | {contrast}: {n_enc} encounters")
                else:
                    print(f"  MISSING: {subject} | {task} | {contrast}")

    return maps

In [4]:
# Load raw beta NIfTI maps
beta_maps = load_beta_maps()

  sub-s03 | nBack | twoBack-oneBack: 5 encounters
  sub-s10 | nBack | twoBack-oneBack: 5 encounters
  sub-s19 | nBack | twoBack-oneBack: 5 encounters
  sub-s29 | nBack | twoBack-oneBack: 5 encounters
  sub-s43 | nBack | twoBack-oneBack: 5 encounters
  sub-s03 | nBack | match-mismatch: 5 encounters
  sub-s10 | nBack | match-mismatch: 5 encounters
  sub-s19 | nBack | match-mismatch: 5 encounters
  sub-s29 | nBack | match-mismatch: 5 encounters
  sub-s43 | nBack | match-mismatch: 5 encounters
  sub-s03 | nBack | task-baseline: 5 encounters
  sub-s10 | nBack | task-baseline: 5 encounters
  sub-s19 | nBack | task-baseline: 5 encounters
  sub-s29 | nBack | task-baseline: 5 encounters
  sub-s43 | nBack | task-baseline: 5 encounters
  sub-s03 | nBack | response_time: 5 encounters
  sub-s10 | nBack | response_time: 5 encounters
  sub-s19 | nBack | response_time: 5 encounters
  sub-s29 | nBack | response_time: 5 encounters
  sub-s43 | nBack | response_time: 5 encounters
  sub-s03 | flanker | inc

In [5]:
# Build per-subject 4D image by concatenating all contrast/encounter images (excluding RT contrast)
subject_imgs = {}
for subj in SUBJECTS:
    imgs = []
    for task in TASKS:
        for contrast in CONTRASTS[task]:
            if contrast == RT_CONTRAST:
                continue
            enc_dict = beta_maps[task][contrast].get(subj, {})
            for enc_key in sorted(enc_dict.keys()):
                imgs.append(enc_dict[enc_key])
    if imgs:
        subject_imgs[subj] = concat_imgs(imgs)

: 

In [ ]:
# Set up brain mask and masker
brain_mask = api.get('MNI152NLin2009cAsym', desc='brain', suffix='mask', resolution=1)
masker = NiftiMasker(mask_img=brain_mask).fit()

In [ ]:
# Split into training subjects and left-out subject (SUBJECTS[-1])
train_subjects = SUBJECTS[:-1]
left_out_subject = SUBJECTS[-1]
template_train = [subject_imgs[s] for s in train_subjects]

In [ ]:
# Compute baseline (voxelwise average of training subjects in masked space)
masked_imgs = [masker.transform(img) for img in template_train]
euclidean_avg = np.mean(masked_imgs, axis=0)

In [ ]:
# Create parcel labels from the first training image
labels = get_labels(template_train[0], n_pieces=150, masker=masker)
dict_alignment = dict(zip(train_subjects, template_train))

In [ ]:
# Fit group template using Procrustes alignment
template_estim = GroupAlignment(method="procrustes", labels=labels)
template_estim.fit(X=dict_alignment, y="template")
procrustes_template = template_estim.template

In [ ]:
# Align left-out subject to the template
left_out_data = masker.transform(subject_imgs[left_out_subject])
pairwise_estim = PairwiseAlignment(method="procrustes", labels=labels).fit(
    left_out_data, procrustes_template
)
predictions_from_template = pairwise_estim.transform(left_out_data)

In [ ]:
# Score: baseline (unaligned average) vs template alignment, both vs left-out subject
average_score = score_voxelwise(left_out_data, euclidean_avg, loss="corr")
template_score = score_voxelwise(left_out_data, predictions_from_template, loss="corr")

In [ ]:
# Plot both scores
average_score_img = masker.inverse_transform(average_score)
template_score_img = masker.inverse_transform(template_score)
baseline_display = plotting.plot_stat_map(
    average_score_img, display_mode="z", vmax=1, cut_coords=[-15, -5]
)
baseline_display.title("Left-out subject correlation with group average")
display = plotting.plot_stat_map(
    template_score_img, display_mode="z", cut_coords=[-15, -5], vmax=1
)
display.title("Aligned subject correlation with Procrustes template")